In [17]:
prev_val

'-'

In [1]:
import os
import pandas as pd

# 1. 병합 대상 폴더 경로
folder_path = "./레서피"  # 현재 폴더 또는 '/mnt/data'
excel_files = sorted([f for f in os.listdir(folder_path) if f.endswith(".xlsx")])

# 2. 기준 시트명 확보 (첫 파일 기준 최대 8개)
first_file_path = os.path.join(folder_path, excel_files[0])
first_xls = pd.ExcelFile(first_file_path)
reference_sheet_names = first_xls.sheet_names[:8]

# 3. 병합 결과 저장 딕셔너리
merged_dataframes = {}

# 4. 시트별 병합 수행
for idx, sheet_name in enumerate(reference_sheet_names):
    all_data_dict = {}

    for file in excel_files:
        file_path = os.path.join(folder_path, file)
        xls = pd.ExcelFile(file_path)
        if idx >= len(xls.sheet_names):
            continue

        df = xls.parse(xls.sheet_names[idx], dtype=str).fillna("-")
        df.columns.values[0] = "항목"
        df = df.set_index("항목")

        for item, row in df.iterrows():
            item = item.strip()

            if item not in all_data_dict:
                all_data_dict[item] = {}
            for col, val in row.items():
                if val != "-":
                    prev_val = all_data_dict[item].get(col, "-")
                    if prev_val == "-":
                        all_data_dict[item][col] = val
                    elif val != prev_val and val not in prev_val.split(" / "):
                        all_data_dict[item][col] = prev_val + " / " + val

    # 병합된 DataFrame 구성
    merged_df = pd.DataFrame.from_dict(all_data_dict, orient="index")
    merged_df.index.name = "항목"
    merged_df = merged_df.reset_index().copy()
    merged_df = merged_df.fillna("-")
    merged_dataframes[sheet_name] = merged_df

# 5. 병합 결과 Excel로 저장
output_file = os.path.join(folder_path, "병합_레시피_항목그대로병합.xlsx")
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    for sheet_name, df in merged_dataframes.items():
        df.to_excel(writer, sheet_name=sheet_name[:31], index=False)

print(f"[완료] 병합 파일 저장 완료 → {output_file}")

[완료] 병합 파일 저장 완료 → ./레서피\병합_레시피_항목그대로병합.xlsx


In [23]:
merged_dataframes.items()

dict_items([('주요식재료',            항목  제육볶음 매운제육볶음 두부제육볶음  김치찌개  된장찌개 청국장찌개 마파두부밥   산라탕  옥수수탕  ...  \
0    돼지고기(목살)  300g   300g   200g  200g     -     -     -     -     -  ...   
1          양파    1개     1개     1개  1/2개  1/2개  1/2개    1개  1/2개  1/4개  ...   
2          대파    2대     2대     2대    1대    1대    1대    2대    2대    2대  ...   
3          마늘    4쪽     5쪽     4쪽    3쪽    3쪽    3쪽    6쪽    3쪽    2쪽  ...   
4          생강    1쪽     1쪽     1쪽     -     -     -   2조각   1조각     -  ...   
..        ...   ...    ...    ...   ...   ...   ...   ...   ...   ...  ...   
294     그릭요거트     -      -      -     -     -     -     -     -     -  ...   
295       베리류     -      -      -     -     -     -     -     -     -  ...   
296      통밀가루     -      -      -     -     -     -     -     -     -  ...   
297        두유     -      -      -     -     -     -     -     -     -  ...   
298     코코넛크림     -      -      -     -     -     -     -     -     -  ...   

    허브티 시저샐러드 탄산음료 소시지 바게트 에스프레소 프로틴볼 그릭요